# Problema do Caixeiro Viajante — Formulação MTZ

**Trabalho AV3 — Modelagem em Programação Matemática (Unifor)**

## Enunciado

O Problema do Caixeiro Viajante (PCV), ou *Traveling Salesman Problem* (TSP), consiste em determinar a **rota de menor custo** que permite a um viajante visitar um conjunto de cidades **exatamente uma vez** e **retornar à cidade de origem**.

Dado um dígrafo completo $D = (V, A)$ com $|V| = n$ cidades e custo $c_{ij}$ em cada arco $(i, j) \in A$, o objetivo é encontrar um **ciclo Hamiltoniano de custo mínimo**.

---

## Formulação — PLI com eliminação de subciclos (Miller-Tucker-Zemlin)

### Variáveis de decisão

- $x_{ij} \in \{0, 1\}$: vale 1 se o arco $(i \to j)$ pertence ao ciclo ótimo.
- $u_i \in \mathbb{Z}$, $i = 2, \ldots, n$: variáveis auxiliares de ordem de visita.

### Função objetivo

$$\min \sum_{(i,j) \in A} c_{ij} \cdot x_{ij}$$

### Restrições

**Cada cidade tem exatamente um arco de saída:**
$$\sum_{j \neq i} x_{ij} = 1, \quad \forall\, i \in V$$

**Cada cidade tem exatamente um arco de entrada:**
$$\sum_{i \neq j} x_{ij} = 1, \quad \forall\, j \in V$$

**Eliminação de subciclos (MTZ):**
$$u_i - u_j + n \cdot x_{ij} \leq n - 1, \quad \forall\, i, j \in V \setminus \{1\},\; i \neq j$$

**Limites das variáveis de ordem:**
$$2 \leq u_i \leq n, \quad \forall\, i \in V \setminus \{1\}$$

**Domínio:**
$$x_{ij} \in \{0, 1\}, \quad u_i \in \mathbb{Z}$$

In [ ]:
!pip install ortools -q

In [ ]:
from ortools.linear_solver import pywraplp
import pandas as pd

## 1. Leitura dos dados de entrada

In [ ]:
# Leitura dos arquivos CSV
df_dados_gerais = pd.read_csv('dados-gerais.csv')
df_dados_arcos = pd.read_csv('dados-arcos.csv')

print('Dados gerais:')
print(df_dados_gerais)
print()
print('Dados dos arcos:')
print(df_dados_arcos)

In [ ]:
# Extrair o numero de vertices
num_vertices = int(df_dados_gerais['num_vertices'][0])

# Criar a lista de vertices (1, 2, ..., n)
vertices = []
for i in range(1, num_vertices + 1):
    vertices.append(i)

# Extrair os arcos do dataframe
arcos = []
for row in df_dados_arcos.itertuples():
    arcos.append((row.origem, row.destino, row.custo))

print(f'Numero de vertices: {num_vertices}')
print(f'Vertices: {vertices}')
print(f'Numero de arcos: {len(arcos)}')
print(f'Arcos esperados (grafo completo): {num_vertices * (num_vertices - 1)}')

## 2. Criação do solver e variáveis de decisão

In [ ]:
# Criar o solver SCIP
solver = pywraplp.Solver.CreateSolver('SCIP')
infinity = solver.infinity()

# Variaveis x[i,j]: vale 1 se o arco (i -> j) esta no ciclo otimo (binaria)
x = {}
for a in arcos:
    i, j = a[0], a[1]
    x[(i, j)] = solver.BoolVar(f'x{i}{j}')

# Variaveis u[i]: ordem de visita da cidade i (inteira, 2 <= u[i] <= n)
# A cidade 1 nao precisa de variavel de ordem (e a origem/retorno)
u = {}
for i in vertices:
    if i != 1:
        u[i] = solver.IntVar(2, num_vertices, f'u{i}')

print(f'Variaveis x[i,j] criadas: {len(x)}')
print(f'Variaveis u[i] criadas: {len(u)}')

## 3. Função objetivo

Minimizar o custo total da rota:
$$\min \sum_{(i,j) \in A} c_{ij} \cdot x_{ij}$$

In [ ]:
# Funcao objetivo: minimizar o custo total do ciclo
objetivo = solver.Objective()
for a in arcos:
    i, j, c = a[0], a[1], a[2]
    objetivo.SetCoefficient(x[(i, j)], c)
objetivo.SetMinimization()

## 4. Restrições

### 4.1 Restrição de saída
Cada cidade tem exatamente 1 arco saindo:
$$\sum_{j \neq i} x_{ij} = 1, \quad \forall\, i \in V$$

In [ ]:
# Restricao de saida: cada cidade tem exatamente 1 arco saindo
for v in vertices:
    restricao = solver.Constraint(1, 1, f'saida_{v}')
    for a in arcos:
        if a[0] == v:
            i, j = a[0], a[1]
            restricao.SetCoefficient(x[(i, j)], 1)

### 4.2 Restrição de entrada
Cada cidade tem exatamente 1 arco entrando:
$$\sum_{i \neq j} x_{ij} = 1, \quad \forall\, j \in V$$

In [ ]:
# Restricao de entrada: cada cidade tem exatamente 1 arco entrando
for v in vertices:
    restricao = solver.Constraint(1, 1, f'entrada_{v}')
    for a in arcos:
        if a[1] == v:
            i, j = a[0], a[1]
            restricao.SetCoefficient(x[(i, j)], 1)

### 4.3 Eliminação de subciclos (MTZ)
$$u_i - u_j + n \cdot x_{ij} \leq n - 1, \quad \forall\, i, j \in V \setminus \{1\},\; i \neq j$$

In [ ]:
# Restricoes MTZ: eliminacao de subciclos
# u[i] - u[j] + n * x[i,j] <= n - 1, para todo i,j != 1
for a in arcos:
    i, j = a[0], a[1]
    if i != 1 and j != 1:
        restricao = solver.Constraint(-infinity, num_vertices - 1, f'mtz_{i}_{j}')
        restricao.SetCoefficient(u[i], 1)
        restricao.SetCoefficient(u[j], -1)
        restricao.SetCoefficient(x[(i, j)], num_vertices)

print(f'Total de restricoes: {solver.NumConstraints()}')

## 5. Modelo de Programação Linear Inteira (formato LP)

In [ ]:
print(solver.ExportModelAsLpFormat(False))

## 6. Resolução e solução ótima

In [ ]:
# Resolver o modelo
status = solver.Solve()

if status == pywraplp.Solver.OPTIMAL:
    custo_total = int(round(objetivo.Value()))

    # Identificar os arcos utilizados na rota otima
    arcos_utilizados = []
    for a in arcos:
        i, j, c = a[0], a[1], a[2]
        if x[(i, j)].solution_value() > 0.5:
            arcos_utilizados.append((i, j, c))

    # Reconstruir a rota a partir dos arcos
    proximo = {}
    for i, j, c in arcos_utilizados:
        proximo[i] = j

    rota = [1]
    atual = 1
    for passo in range(num_vertices):
        atual = proximo[atual]
        rota.append(atual)

    # Exibir resultados
    print('Solucao otima encontrada.')
    print()
    print(f'Rota: {" -> ".join(str(v) for v in rota)}')
    print(f'Custo total: {custo_total}')
    print()
    print('Arcos utilizados:')
    for i, j, c in arcos_utilizados:
        print(f'  {i} -> {j}  (custo {c})')
    print()

    # Exibir ordem de visita (variaveis u)
    print('Ordem de visita (variaveis u):')
    print(f'  u[1] = 1  (origem)')
    for i in vertices:
        if i != 1:
            print(f'  u[{i}] = {int(round(u[i].solution_value()))}')
else:
    print('O problema nao tem solucao otima.')